# 📘 Notebook 01 · 量化入门 + 多因子模型

> 量化交易的"标准武器"是多因子模型。把这一节吃透，你已经超过 70% 自学者。

**本 notebook 学完你会：**

- 知道量化交易在解决什么问题、有哪些类型
- 能用 Python 算 Sharpe、最大回撤、IC、ICIR 等核心指标
- 能从零写一个单因子（动量 / 价值 / 反转）并做检验
- 能把多个因子合成一个综合得分，分层回测
- 能用 cvxpy 做带约束的组合优化

**预计学习时间：8-12 小时**（可以拆 3-4 天）

## 1. 量化交易是什么

### 1.1 一句话定义

> 把交易决策**用数学模型与代码表达**、**用历史数据验证**、**用计算机自动执行**的完整工程。

它不是占卜，也不是技术指标的简单堆砌，本质是**金融 + 统计 + 工程**的交叉学科。

### 1.2 量化 vs 主观

| 维度 | 主观交易 | 量化交易 |
|---|---|---|
| 决策依据 | 经验、消息、感觉 | 数据、模型、统计显著性 |
| 执行方式 | 手动下单 | 代码自动下单 |
| 可复现性 | 差 | 强 |
| 可扩展性 | 盯不过来 | 几千个标的并行 |
| 主要风险 | 情绪、过度交易 | 过拟合、模型失效 |

### 1.3 量化的五种主要类型

| 类型 | 思想 | 代表方法 | 周期 |
|---|---|---|---|
| **趋势跟踪 (CTA)** | 强者恒强 | 双均线、突破、海龟 | 天-周 |
| **截面 / 统计套利** | 横截面相对强弱 | 多因子、配对交易 | 天-月 |
| **均值回归** | 价格围绕均值波动 | Bollinger、Z-score | 分钟-天 |
| **事件驱动** | 信息→价格冲击 | 并购套利、PEAD | 事件期 |
| **做市 / 高频** | 提供流动性赚价差 | 限价单簿建模 | 毫秒-秒 |

本 notebook 主要聚焦 **截面 / 统计套利** 中的**多因子选股**——它是国内私募最主流的策略类型。

## 2. 准备数据：用模拟数据先把流程跑通

学量化最大的坑：**一上来就纠结数据从哪来**。

正确的顺序是：**先用模拟数据跑通流程**，确认代码无误，再接真实数据。下面我们生成一份模拟的股票数据（300 只股票、5 年日频）。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# ---------- 生成模拟数据 ----------
n_stocks = 300
n_days   = 1250  # 约 5 年
dates    = pd.bdate_range('2020-01-01', periods=n_days)
tickers  = [f'S{i:03d}' for i in range(n_stocks)]

# 给每只股票一个"真实因子暴露"，让模拟数据有可学习的结构
true_alpha = np.random.randn(n_stocks) * 0.0003           # 真实超额收益
factor_loading = np.random.randn(n_stocks, 3) * 0.5        # 3 个因子暴露
market_factor = np.random.randn(n_days) * 0.012            # 市场因子日收益
style_factors = np.random.randn(n_days, 3) * 0.008         # 3 个风格因子

# 个股日收益 = α + β·market + 因子暴露·因子收益 + 特质噪声
idio = np.random.randn(n_days, n_stocks) * 0.018
beta = 0.8 + np.random.rand(n_stocks) * 0.6                # β 在 0.8~1.4
returns = (true_alpha
           + np.outer(market_factor, beta)
           + style_factors @ factor_loading.T
           + idio)

returns = pd.DataFrame(returns, index=dates, columns=tickers)
price = 10 * (1 + returns).cumprod()                       # 累乘成价格

print(f'数据形状: {price.shape}  (天数 × 股票数)')
print(f'\n前 5 行 5 列:')
price.iloc[:5, :5]

In [ ]:
# 画几只股票的累计收益看一下
fig, ax = plt.subplots(figsize=(10, 4))
for t in tickers[:5]:
    (price[t] / price[t].iloc[0]).plot(ax=ax, label=t, alpha=0.8)
ax.set_title('5 只模拟股票的归一化价格走势')
ax.legend(loc='best')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. 必会的绩效指标

在做任何策略前，先把"怎么评估"搞清楚。下面这几个指标，你应该闭着眼能默写。

In [ ]:
def annualize_return(returns, periods=252):
    """年化收益率"""
    return (1 + returns.mean()) ** periods - 1

def annualize_vol(returns, periods=252):
    """年化波动率"""
    return returns.std() * np.sqrt(periods)

def sharpe_ratio(returns, rf=0.03, periods=252):
    """夏普比率（年化）"""
    excess = returns - rf / periods
    return excess.mean() / returns.std() * np.sqrt(periods)

def max_drawdown(returns):
    """最大回撤"""
    cum = (1 + returns).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd.min()

def calmar_ratio(returns, periods=252):
    """卡玛比率 = 年化收益 / |最大回撤|"""
    return annualize_return(returns, periods) / abs(max_drawdown(returns))

def sortino_ratio(returns, rf=0.03, periods=252):
    """索提诺比率：只用下行波动"""
    excess = returns - rf / periods
    downside = returns[returns < 0].std() * np.sqrt(periods)
    return excess.mean() * periods / downside if downside > 0 else np.nan

# 拿等权组合（每天等权持有所有股票）演示
eq_weight_ret = returns.mean(axis=1)

print(f'年化收益: {annualize_return(eq_weight_ret):.2%}')
print(f'年化波动: {annualize_vol(eq_weight_ret):.2%}')
print(f'夏普比率: {sharpe_ratio(eq_weight_ret):.3f}')
print(f'索提诺  : {sortino_ratio(eq_weight_ret):.3f}')
print(f'最大回撤: {max_drawdown(eq_weight_ret):.2%}')
print(f'卡玛比率: {calmar_ratio(eq_weight_ret):.3f}')

**关键理解：**

- **Sharpe**：风险调整后收益。> 1 可接受，> 2 优秀。注意这是**年化**值。
- **最大回撤**：心理承受关键。Sharpe 3 + 回撤 40% 的策略，没几个人扛得住。
- **Calmar**：回撤敏感的衡量。私募内部经常用这个看长期表现。
- **Sortino**：只惩罚向下波动，更符合投资者的真实痛感（涨得多没人嫌）。

**常见错误：**

- ❌ 用日收益均值 × 252 算"年化"，正确是 `(1+mean)^252 - 1`
- ❌ 算 Sharpe 忘记减无风险利率
- ❌ 把"季度收益 12%"叫"年化 48%"（不能简单乘 4）

### 🎯 练习

写一个 `summary(returns)` 函数，输入一个日收益率 Series，返回一个 dict 包含：
- 年化收益、年化波动、Sharpe、Sortino、最大回撤、Calmar、胜率（正收益日占比）

然后用它对比"等权组合"和"前 50 只股票等权组合"。

In [ ]:
# 在这里写你的答案


## 4. 第一个因子：动量（Momentum）

### 4.1 思想

**强者恒强**——过去 6-12 个月涨得好的股票，未来短期还会继续涨。这是 Jegadeesh & Titman (1993) 发现的经典异象，**至今在多数市场仍然有效**（虽然时强时弱）。

### 4.2 公式

20 日动量因子定义：

$$\text{MOM}_{i,t} = \frac{P_{i,t-1}}{P_{i,t-21}} - 1$$

注意要**滞后一天**（用昨天的价格算今天的因子），否则就是未来函数。

In [ ]:
def compute_momentum(price, window=20, lag=1):
    """
    计算动量因子：过去 window 日累计收益
    lag: 滞后天数，至少为 1，避免使用 t 日收盘价决定 t 日开盘下单
    """
    return price.pct_change(window).shift(lag)

mom20 = compute_momentum(price, window=20)
mom60 = compute_momentum(price, window=60)

print(f'mom20 形状: {mom20.shape}')
print(f'某一天的因子分布：')
mom20.iloc[100].describe()

In [ ]:
# 看看因子值的分布（应该接近正态，有点尖峰）
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
mom20.iloc[100].hist(bins=50, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('20 日动量 · 截面分布（某一天）')
axes[0].axvline(0, color='red', linewidth=1)

mom20.mean(axis=1).plot(ax=axes[1], color='darkgreen', alpha=0.7)
axes[1].set_title('20 日动量 · 横截面均值随时间变化')
axes[1].axhline(0, color='red', linewidth=0.8)
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. IC 检验：判断因子好不好

### 5.1 什么是 IC

**Information Coefficient（信息系数）**：因子值与未来收益的**横截面相关性**。

每天算一个 IC：

$$\text{IC}_t = \text{corr}(f_{i,t}, r_{i,t+1})$$

其中 $f_{i,t}$ 是股票 $i$ 在 $t$ 日的因子值，$r_{i,t+1}$ 是它未来 1 日收益。

实际中**几乎都用 Spearman（秩相关）**而不是 Pearson，因为因子值常常有重尾。

### 5.2 关键指标

- **IC 均值**：> 0.03 算可用，> 0.05 不错，> 0.08 非常好
- **IC 标准差**：越小越稳定
- **ICIR**：`IC均值 / IC标准差`，> 0.5 算好。这是**最重要**的因子评价指标
- **IC > 0 胜率**：超过 55% 才算稳定方向

In [ ]:
def factor_ic(factor, fwd_returns, method='spearman'):
    """逐日计算横截面 IC"""
    # 对齐索引，并且只在两边都非空的位置算
    ic_series = factor.corrwith(fwd_returns, axis=1, method=method)
    return ic_series

# 未来 1 日收益
fwd_ret_1d = returns.shift(-1)

ic = factor_ic(mom20, fwd_ret_1d)

def ic_summary(ic, periods=252):
    s = {
        'IC 均值': ic.mean(),
        'IC 标准差': ic.std(),
        'ICIR (年化)': ic.mean() / ic.std() * np.sqrt(periods),
        'IC > 0 胜率': (ic > 0).mean(),
        't 值': ic.mean() / (ic.std() / np.sqrt(len(ic))),
        '样本数': len(ic.dropna()),
    }
    return pd.Series(s)

print('20 日动量因子 · IC 检验结果\n')
print(ic_summary(ic).round(4))

In [ ]:
# 画 IC 时序图 + 累计 IC
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ic.rolling(20).mean().plot(ax=axes[0], color='steelblue', alpha=0.9)
axes[0].axhline(0, color='red', linewidth=0.8)
axes[0].set_title('20 日动量 · IC 的 20 日滚动均值')
axes[0].grid(alpha=0.3)

ic.cumsum().plot(ax=axes[1], color='darkgreen')
axes[1].set_title('IC 累计值（向上=因子稳定有效）')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**怎么看这两张图：**

- 第一张图（滚动 IC）：在 0 线上下来回波动是正常的，关键看**有没有持续偏一侧**。
- 第二张图（累计 IC）：**单调向上**说明因子稳定有效，**忽上忽下**说明因子不稳定，**先上后下**说明因子已经失效。

**模拟数据里 IC 看起来不一定漂亮**，因为我们没有特意把动量做强。下一节我们做分层回测，能更直观看到因子的效力。

### 🎯 练习

做以下三个练习：

1. 计算 60 日动量因子（mom60）的 IC，与 20 日动量比较，哪个更好？
2. 实现"反转因子"：过去 5 日收益取负号。看看在我们的模拟数据上 IC 是正还是负？
3. 把所有 mom20 的因子值在每天内做 zscore 标准化，再算 IC——结果会不会变？（理论上不会，思考为什么）

In [ ]:
# 在这里写你的答案


## 6. 分层回测：把因子变成钱

光看 IC 还不够直观。**分层回测**才是金融行业最常用的因子可视化方式：

1. 每一天，按因子值把股票分成 5 组（quintile）
2. 每组等权买入，下一天看收益
3. 长期看，因子值最高的组（Q5）应该比最低的组（Q1）涨得多
4. **多空组合**：买 Q5、卖空 Q1，看净值曲线

In [ ]:
def quantile_backtest(factor, fwd_returns, n_groups=5):
    """
    分层回测
    factor:      (date × stock) 因子值
    fwd_returns: (date × stock) 未来 1 日收益
    返回每个分组的日收益序列
    """
    # 每天按因子值分组（rank 后切分位）
    ranks = factor.rank(axis=1, pct=True)
    group_rets = {}
    for g in range(n_groups):
        lo, hi = g / n_groups, (g + 1) / n_groups
        mask = (ranks > lo) & (ranks <= hi)
        # 这一组的等权收益（每天）
        group_rets[f'Q{g+1}'] = (fwd_returns * mask).sum(axis=1) / mask.sum(axis=1).replace(0, np.nan)
    df = pd.DataFrame(group_rets)
    df['Q5-Q1'] = df['Q5'] - df['Q1']    # 多空组合
    return df

group_rets = quantile_backtest(mom20, fwd_ret_1d, n_groups=5)
print(group_rets.head())
print()
print('各分组年化收益：')
print(((1 + group_rets).prod() ** (252/len(group_rets)) - 1).round(4))

In [ ]:
# 画分层净值
cum = (1 + group_rets).cumprod()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for col in ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']:
    cum[col].plot(ax=axes[0], label=col, alpha=0.85)
axes[0].set_title('5 分位组净值曲线')
axes[0].legend(); axes[0].grid(alpha=0.3)

cum['Q5-Q1'].plot(ax=axes[1], color='red', linewidth=1.5)
axes[1].set_title('多空组合（Q5 - Q1）')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'多空组合 · 年化收益: {annualize_return(group_rets["Q5-Q1"]):.2%}')
print(f'多空组合 · 年化波动: {annualize_vol(group_rets["Q5-Q1"]):.2%}')
print(f'多空组合 · 夏普    : {sharpe_ratio(group_rets["Q5-Q1"]):.3f}')

**怎么看分层图：**

- **理想形态**：Q5 > Q4 > Q3 > Q2 > Q1，**单调**——这是因子最看重的性质
- 如果只是 Q5 比 Q1 高、中间组乱序，因子可能只在极端组有效，**不稳健**
- 多空净值**长期向上、回撤小**是好因子
- 多空净值**先涨后跌**说明因子在样本后期失效，要警惕

**实战提醒：**

- 真实数据中，多空组合需要考虑 **A 股做空成本**（融券利率 6-10%）
- 中国市场常用 **多头组合相对指数** 来评价，而不是真正做空
- 月度调仓比日度调仓更现实（手续费更可控）

### 🎯 练习

做以下两个练习：

1. 把分层回测改成 **10 分组**，画出 Q10-Q1 多空净值。多空效果比 5 分组好还是差？
2. 实现 **月频调仓**：每月最后一个交易日按因子排序，持有到下月末。比较月频 vs 日频的换手率和净收益。

In [ ]:
# 在这里写你的答案


## 7. 因子预处理：去极值 + 中性化 + 标准化

原始因子直接拿来用问题很多：

- **极端值**：某个公司可能因为重组导致动量值 = 500%
- **行业偏离**：动量在某些行业总是高（如科技），可能只是行业暴露
- **市值偏离**：小盘股波动天然大，所有动量值都被市值"染色"

**三步预处理是行业标准：**

1. **去极值**（Winsorize）：把超过 ±3σ 或 MAD（中位数绝对偏差）阈值的值截断
2. **中性化**：对市值、行业等做回归，取残差
3. **标准化**：zscore，让所有因子在同一量级

In [ ]:
def winsorize_mad(s, n=5):
    """用 MAD 去极值（比 ±3σ 更鲁棒）"""
    med = s.median()
    mad = (s - med).abs().median()
    return s.clip(lower=med - n*1.4826*mad, upper=med + n*1.4826*mad)

def standardize(s):
    return (s - s.mean()) / s.std()

def preprocess_factor(factor):
    """逐日去极值 + 标准化"""
    return factor.apply(lambda row: standardize(winsorize_mad(row)), axis=1)

mom20_clean = preprocess_factor(mom20)

# 对比预处理前后
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
mom20.iloc[100].hist(bins=50, ax=axes[0], color='steelblue', alpha=0.7)
axes[0].set_title('预处理前')
mom20_clean.iloc[100].hist(bins=50, ax=axes[1], color='darkgreen', alpha=0.7)
axes[1].set_title('预处理后（去极值 + zscore）')
plt.tight_layout(); plt.show()

**中性化**怎么做（真实数据用）：

```python
# 对每一天，把因子对市值、行业哑变量做 OLS 回归，取残差
import statsmodels.api as sm

def neutralize(factor_row, mcap_row, industry_dummies):
    X = pd.concat([np.log(mcap_row), industry_dummies], axis=1)
    X = sm.add_constant(X.dropna())
    y = factor_row.loc[X.index]
    resid = sm.OLS(y, X).fit().resid
    return resid.reindex(factor_row.index)
```

我们的模拟数据没有市值和行业，所以这里只演示去极值 + 标准化。真实做法在实战项目里展开。

## 8. 多因子合成

单个因子的 IC 一般 0.03~0.05，组合多个因子可以把 ICIR 提到 0.8+。

### 8.1 三种合成方法

| 方法 | 做法 | 优缺点 |
|---|---|---|
| **等权** | 直接平均 | 简单稳健，但忽略因子强弱差异 |
| **IC 加权** | 按历史 IC 均值加权 | 反映有效性，对窗口敏感 |
| **最大化 ICIR** | 求解优化问题：`max w'·μ / sqrt(w'·Σ·w)` | 理论最优，但容易过拟合 |

初学者建议**先用等权 / IC 加权**，等数据多了再上 ICIR 优化。

### 8.2 实现三个因子并合成

In [ ]:
# 我们简单造三个因子：20日动量 + 5日反转 + 20日波动率（取负）
def factor_momentum(price, w=20): return price.pct_change(w).shift(1)
def factor_reversal(price, w=5):  return -price.pct_change(w).shift(1)
def factor_lowvol(price, w=20):
    ret = price.pct_change()
    return -ret.rolling(w).std().shift(1)

f1 = preprocess_factor(factor_momentum(price))
f2 = preprocess_factor(factor_reversal(price))
f3 = preprocess_factor(factor_lowvol(price))

# --- 看每个因子的 ICIR ---
for name, f in [('mom20', f1), ('rev5', f2), ('lowvol20', f3)]:
    ic = factor_ic(f, fwd_ret_1d)
    print(f'{name:10s}  ICIR = {ic.mean()/ic.std()*np.sqrt(252):7.3f}   IC均值 = {ic.mean():7.4f}')

In [ ]:
# 等权合成
composite_eq = (f1 + f2 + f3) / 3

# IC 加权合成：用过去 60 日 IC 均值作为权重
def ic_weighted_composite(factors_dict, fwd_ret, ic_window=60):
    """
    factors_dict: {'mom': df, 'rev': df, 'vol': df}
    动态用过去 60 日 IC 均值作为权重，rolling 计算
    """
    ic_dict = {name: factor_ic(f, fwd_ret) for name, f in factors_dict.items()}
    # 每个因子的 IC 滚动均值（要 shift(1)，避免未来函数）
    ic_w = pd.DataFrame({n: ic.rolling(ic_window).mean().shift(1) for n, ic in ic_dict.items()})
    # 对每一天，把权重归一化
    ic_w = ic_w.div(ic_w.abs().sum(axis=1), axis=0)
    # 合成：每天按权重加权 factors
    composite = sum(ic_w[n].values[:, None] * factors_dict[n] for n in factors_dict)
    return pd.DataFrame(composite, index=f1.index, columns=f1.columns)

composite_icw = ic_weighted_composite({'mom': f1, 'rev': f2, 'vol': f3}, fwd_ret_1d)

print('==== 合成因子对比 ====')
for name, f in [('等权合成', composite_eq), ('IC加权合成', composite_icw)]:
    ic = factor_ic(f, fwd_ret_1d)
    print(f'{name:12s}  ICIR = {ic.mean()/ic.std()*np.sqrt(252):7.3f}   IC均值 = {ic.mean():7.4f}')

In [ ]:
# 对合成因子做分层回测
gr_eq  = quantile_backtest(composite_eq, fwd_ret_1d)
gr_icw = quantile_backtest(composite_icw, fwd_ret_1d)

fig, ax = plt.subplots(figsize=(11, 4))
(1 + gr_eq['Q5-Q1']).cumprod().plot(ax=ax, label='等权合成 · Q5-Q1', color='steelblue')
(1 + gr_icw['Q5-Q1']).cumprod().plot(ax=ax, label='IC加权 · Q5-Q1', color='darkred')
ax.set_title('合成因子多空净值对比')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 组合优化：从打分到持仓

分层回测好看，但真实组合不会无脑等权。**真实做法是带约束的优化问题**：

$$
\begin{aligned}
\max_w \quad & w^\top \hat\mu - \frac{\lambda}{2} w^\top \Sigma w \\
\text{s.t.} \quad & \sum w_i = 1 \\
& 0 \le w_i \le w_{\max} \quad (\text{单票上限}) \\
& |w_i - w_{i,bench}| \le \delta \quad (\text{行业偏离约束}) \\
\end{aligned}
$$

- $\hat\mu$：因子得分（预期收益代理）
- $\Sigma$：协方差矩阵
- $\lambda$：风险厌恶系数
- $w_{\max}$：单只股票上限（控制集中度）

下面用 **cvxpy** 求解。如果环境里没装，先 `pip install cvxpy`。

In [ ]:
# 简化版组合优化：用某一天的因子得分，求最优权重
# 这里只演示某一日的优化逻辑，真实应用要做日度滚动

try:
    import cvxpy as cp
    HAS_CVXPY = True
except ImportError:
    HAS_CVXPY = False
    print('未安装 cvxpy，跳过优化演示')
    print('安装: pip install cvxpy')

if HAS_CVXPY:
    # 取某一天的因子得分作为 alpha
    date_pick = composite_icw.index[800]
    alpha = composite_icw.loc[date_pick].dropna().values
    n = len(alpha)

    # 用过去 60 日收益估协方差
    hist = returns.loc[:date_pick].iloc[-60:][composite_icw.columns]
    sigma = hist.cov().values

    # 求解
    w = cp.Variable(n)
    risk_aversion = 5.0
    objective = cp.Maximize(alpha @ w - 0.5 * risk_aversion * cp.quad_form(w, cp.psd_wrap(sigma)))
    constraints = [
        cp.sum(w) == 1,        # 满仓
        w >= 0,                 # 不允许做空（A股）
        w <= 0.05,              # 单票上限 5%
    ]
    cp.Problem(objective, constraints).solve()

    weights = pd.Series(w.value, index=composite_icw.columns)
    weights = weights[weights > 1e-4].sort_values(ascending=False)

    print(f'优化结果：选出 {len(weights)} 只股票')
    print(f'最大持仓: {weights.max():.2%}')
    print(f'最小持仓: {weights.min():.2%}')
    print(f'前 10 只:\n{weights.head(10).round(4)}')

**实战提醒：**

- 协方差矩阵不能直接用样本协方差（300 只股票 × 60 天 = 矩阵不满秩）。真实做法用**因子风险模型**（Barra）或 **shrinkage**（Ledoit-Wolf）
- 风险厌恶系数 λ 通常通过历史 walk-forward 调出来，不是拍脑袋
- 行业 / 风格中性约束是大头基金的标配，**不做中性化的因子组合很容易表面好看实际靠风格漂移赚钱**

### 🎯 练习

三个练习：

1. 把单票上限改成 3%，对比组合的多样化程度（持仓只数、HHI 指数）
2. 增加一个约束：组合波动率不能超过 15%（用 `cp.quad_form(w, sigma) <= 0.15**2/252`，注意单位）
3. 用 Ledoit-Wolf 收缩估计协方差矩阵（`sklearn.covariance.LedoitWolf`），看优化结果稳定性是否提升

In [ ]:
# 在这里写你的答案


## 10. 把所有东西串起来：完整的因子研究流程

一个标准的因子研究 pipeline 包含以下 8 步。**这是面试时被问"你怎么研究因子"的标准答案**：

In [ ]:
def factor_research_pipeline(price, factor_func, **kwargs):
    """
    一个标准的因子研究流程
    """
    # Step 1: 计算原始因子
    raw_factor = factor_func(price, **kwargs)

    # Step 2: 预处理（去极值 + 标准化）
    factor = preprocess_factor(raw_factor)

    # Step 3: 计算未来收益（一定要 shift 防止未来函数）
    fwd_ret = price.pct_change().shift(-1)

    # Step 4: IC 检验
    ic = factor_ic(factor, fwd_ret)
    ic_stat = ic_summary(ic)

    # Step 5: 分层回测
    group_rets = quantile_backtest(factor, fwd_ret, n_groups=5)

    # Step 6: 多空组合统计
    ls = group_rets['Q5-Q1']
    ls_stat = {
        '年化收益': annualize_return(ls),
        '年化波动': annualize_vol(ls),
        '夏普    ': sharpe_ratio(ls),
        '最大回撤': max_drawdown(ls),
    }

    # Step 7: 画图
    fig, axes = plt.subplots(2, 2, figsize=(13, 7))
    ic.rolling(20).mean().plot(ax=axes[0, 0], color='steelblue')
    axes[0, 0].axhline(0, color='r', lw=0.8); axes[0, 0].set_title('IC 滚动均值'); axes[0, 0].grid(alpha=0.3)

    ic.cumsum().plot(ax=axes[0, 1], color='darkgreen')
    axes[0, 1].set_title('累计 IC'); axes[0, 1].grid(alpha=0.3)

    cum_groups = (1 + group_rets[['Q1','Q2','Q3','Q4','Q5']]).cumprod()
    for col in cum_groups.columns:
        cum_groups[col].plot(ax=axes[1, 0], label=col, alpha=0.85)
    axes[1, 0].set_title('5 分位净值'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

    (1 + ls).cumprod().plot(ax=axes[1, 1], color='red')
    axes[1, 1].set_title('多空组合净值'); axes[1, 1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()

    # Step 8: 打印报告
    print('=' * 50)
    print('IC 检验:')
    print(ic_stat.round(4))
    print('\n多空组合表现:')
    for k, v in ls_stat.items():
        print(f'  {k}: {v:.4f}')
    print('=' * 50)
    return factor, ic, group_rets

# 跑一遍
_ = factor_research_pipeline(price, factor_momentum, w=20)

### 🎯 练习

**综合大练习**：

设计你自己的因子，跑通完整流程：

1. **想一个因子**：可以是技术类（如"过去 N 日最高价距离当前价"）、波动类、形态类。说出它的经济学逻辑（为什么有效）。
2. 把它实现成函数 `def my_factor(price, ...)`，输出 (date × stock) DataFrame。
3. 跑 `factor_research_pipeline(price, my_factor)`。
4. 写一段话评估：**这个因子值不值得用**？理由是什么？

**评分标准**（自己打）：
- ICIR > 0.3 算合格
- 多空 Sharpe > 0.5 算合格
- 多空净值单调向上无明显回撤期 = 优秀

In [ ]:
# 在这里写你的答案


## 11. 本节小结

**你现在应该会的：**

- ✅ 用 numpy / pandas 算所有核心绩效指标
- ✅ 从原始价格数据计算因子
- ✅ 因子预处理（去极值、标准化）
- ✅ IC / ICIR 检验
- ✅ 分层回测
- ✅ 多因子合成（等权 / IC 加权）
- ✅ 用 cvxpy 做组合优化

**还差什么（在 Notebook 02 / 03 补）：**

- ⬜ 趋势跟踪、配对交易、ML 选股
- ⬜ 用 Backtrader / Qlib 等专业框架做回测
- ⬜ 严格的样本外检验（walk-forward、purged k-fold）
- ⬜ 真实数据接入（akshare / tushare）
- ⬜ 实盘对接 vnpy

**重要心态：**

> 量化的核心不是"挖出秘密公式"，而是**把每一步研究做扎实，对每一个结果保持怀疑**。

> 你今天看到的 ICIR = 0.8 的因子，明天可能就 0.1。所有研究都需要持续监控，**没有一劳永逸的策略**。

---

**下一节预告：`02_量化策略_回测实战.ipynb`**

我们会讲：CTA 趋势跟踪、配对交易、机器学习选股，以及怎么用专业框架做严格的样本外回测。